In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDac-AGL/AgentDaC/notebooks


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerBase
import pathlib
from src.utils.chat_template import resolve_chat_template


MODEL_NAME = "Qwen/Qwen3.5-4B"
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer: PreTrainedTokenizerBase = AutoTokenizer.from_pretrained(MODEL_NAME)


CONVERSATION = [
    {
        "role": "system",
        "content": "You are a helpful assistant that excels at solving tasks.\nYou solve a task by reasoning and, when it helps, breaking it into smaller sub-tasks.\n\nKeep your internal reasoning brief and to the point.\n\nEach round, do exactly one of:\n- Delegate a focused sub-task using one of the delegation tools available to you (e.g. `create_new_sub_agent`); you will receive the sub-agent's answer back and can continue.\n- Or, when you have what you need, finish by calling the `submit_answer` tool with your final answer in the exact format the task requests.\n\nIMPORTANT: The ONLY way to finish is to call the `submit_answer` tool. Do NOT write your final answer as plain text \u2014 put it in the `answer` argument of `submit_answer`, with no extra commentary. `submit_answer` ends the task.\n\nA sub-agent is a blank slate: it has NO memory and NO knowledge of this conversation or the original task. Anything you delegate must be completely self-contained - restate the full position (including the complete board/FEN) and exactly what the sub-agent should return.\n\nAvoid these pitfalls:\n- Do NOT delegate the whole task to a single sub-agent. Delegation only helps if you split the problem into smaller, focused pieces and combine their results yourself.\n- Do NOT over-decompose: delegate only when a sub-task genuinely helps.\n- Spend your effort on good decomposition and a correct final answer, not on long reasoning.",
    },
    {
        "role": "user",
        "content": "Task:\nYou are given a chess position. Find the best legal move for the side to move. Important: don't reason too long, you have token limits.Use sub-tasks to break down the problem if needed and break down the reasoning into smaller steps.\n\nCurrent FEN:\nrn2kb1r/p3qp1p/3p1p2/1p1pp3/4P3/1QPP1P2/PP3P1P/RN2K2R w KQkq - 0 11\n\nSide to move:\nWhite\n\nBoard, White at bottom (uppercase = White, lowercase = Black, '.' = empty square):\n8  r n . . k b . r\n7  p . . . q p . p\n6  . . . p . p . .\n5  . p . p p . . .\n4  . . . . P . . .\n3  . Q P P . P . .\n2  P P . . . P . P\n1  R N . . K . . R\n   a b c d e f g h\n\nLegal moves (UCI):\na2a3 a2a4 b1a3 b1d2 b3a3 b3a4 b3b4 b3b5 b3c2 b3c4 b3d1 b3d5 c3c4 d3d4 e1d1 e1d2 e1e2 e1f1 e1g1 e4d5 f3f4 h1f1 h1g1 h2h3 h2h4\n\nFinal answer format:\nThe final answer must be exactly one legal move in UCI notation. The final move should appear within \\boxed{...} as a single move. ",
    },
    {
        "role": "user",
        "content": "<controller_state>\nremaining_rounds: 4\nremaining_delegations: 2\nallowed_tools: ['create_new_sub_agent', 'submit_answer']\n</controller_state>",
    },
    {
        "role": "assistant",
        "content": "I need to find the best legal move for White from this chess position. Let me first understand the board position:\n\nFrom the FEN: `rn2kb1r/p3qp1p/3p1p2/1p1pp3/4P3/1QPP1P2/PP3P1P/RN2K2R w KQkq - 0 11`\n\nLet me decode this to understand the position:\n- Piece layout (FEN is from white's perspective, so bottom rows are ranks 1-2, top ranks 7-8)\n- Side to move: white\n- Castling: white can castle K or Q, black cannot castle\n\nLooking at the board representation given:\n```\n8  r n . . k b . r   (rank 8: rook on a8, knight on b8, king on e8, bishop on f8, rook on h8)\n7  p . . . q p . p   (rank 7: pawn on a7, queen on g7, pawn on g7? wait let me recheck)\n```\n\nWait, let me look at the piece layout more carefully:\n```\nBoard, White at bottom (uppercase = White, lowercase = Black, '.' = empty square):\n8  r n . . k b . r\n7  p . . . q p . p\n6  . . . p . p . .\n5  . p . p p . . .\n4  . . . . P . . .\n3  . Q P P . P . .\n2  P P . . . P . P\n1  R N . . K . . R\n   a b c d e f g h\n```\n\nSo rank 1 (bottom): R on a1, N on b1, K on e1, R on h1\nRank 2: P on a2, P on b2, P on f2, P on h2\n\n\nRank 3: Q on a3, P on b3, P on c3, P on f3\nRank 4: Empty except maybe... looking at FEN again: `4  . . . . P . . .` so P on e4\nRank 5: `5  . p . p p . . .` so pawns on b5, d5, e5\nRank 6: `6  . . . p . p . .` so pawns on d6, f6\nRank 7: `7  p . . . q p . p` so pawns on a7, g7, h7 and Queen on g7\nRank 8: `8  r n . . k b . r` so pieces on a8, b8, e8, f8, h8\n\nI'm tracking the complex board setup, noting key pieces and their positions across different ranks. The middle game phase suggests strategic planning is crucial. I'll analyze potential moves systematically, focusing on immediate tactical opportunities and piece development.\n\nThe opening moves reveal promising candidates: developing knights and bishops while maintaining central control. I'll prioritize moves that improve piece placement and restrict opponent's options.\n\nMy initial evaluation suggests three strong positional developments: moving the queen to an active square, advancing the knight, and strengthening central tension. Each move has specific strategic benefits for controlling the center and preparing for future attacks.\n\nThe next step involves examining these candidate moves through concrete tactical analysis, considering material balance and positional advantages.\n</think>\n\n<tool_call>\n<function=create_new_sub_agent>\n<parameter=text>\nFind the best legal move for White in this chess position.\n\nComplete position FEN: rn2kb1r/p3qp1p/3p1p2/1p1pp3/4P3/1QPP1P2/PP3P1P/RN2K2R w KQkq - 0 11\n\nKey details:\n- White to move\n- White can castle short or long\n- Material appears balanced\n\nYour task: Analyze all legal moves from the list of legal moves provided and determine which is the BEST move tactically and strategically.\n\nConsider:\n1. Is there any tactic (fork, pin, discovered attack)?\n2. Does it control key central squares?\n3. Does it develop a piece properly?\n4. Does it prevent counterplay?\n5. Is it forcing?\n\nLegal moves (UCI notation):\na2a3 a2a4 b1a3 b1d2 b3a3 b3a4 b3b4 b3b5 b3c2 b3c4 b3d1 b3d5 c3c4 d3d4 e1d1 e1d2 e1e2 e1f1 e1g1 e4d5 f3f4 h1f1 h1g1 h2h3 h2h4\n\nReturn your final answer as exactly one legal move in UCI format (e.g., \"e2e4\"), with reasoning if helpful but clearly identify the single best move.\n</parameter>\n</function>\n</tool_call>",
    },
    {
        "role": "tool",
        "content": "e1g1",
        "tool_call_id": "chatcmpl-tool-82a20c2abbf1a869",
    },
    {
        "role": "user",
        "content": "<controller_state>\nremaining_rounds: 3\nremaining_delegations: 1\nallowed_tools: ['create_new_sub_agent', 'send_message_to_current_sub_agent', 'submit_answer']\n</controller_state>",
    },
    {
        "role": "assistant",
        "content": "The sub-agent returned e1g1 as the answer. Let me verify this makes sense in the position:\n\nPosition analysis:\n- White to move\n- Queen is on a3 (rank 3) - this seems unusual for an opening but that's the position\n- Looking at the board again from my earlier analysis... wait, I need to re-check the FEN vs the board representation.\n\nActually, looking at the board representation given:\n```\n3  . Q P P . P . .\n```\nThis shows Q on a3 (column a, row 3), which matches what I noted.\n\nThe move e1g1 means moving the king from e1 to g1. That would be castling long (queenside)? No wait, long castle would be O-O-O which moves king to c1.\n\nWait, e1g1 is moving the king from e1 to g1 with rook from h1. This is actually castling queenside? No, standard notation:\n- K = King side castling (king to g1)\n- Q = Queen side castling (king to c1)\n\nSo e1g1 would be kingside castling if we use algebraic notation directly, but the task says UCI notation where files are letters and ranks are digits. So e1g1 means king on e1 goes to g1, which is indeed kingside castling.\n\nIs kingside castling good here? With Black having pieces on f8 and the position being somewhat open, castling might expose the king to potential attack. But let me trust the analysis.\n\nActually wait - I should double-check. The sub-agent only returned \"e1g1\" without much explanation, but it used its analysis tools presumably.\n\nHowever, let me reconsider whether there might be better moves. Looking at some other options:\n- b1d2 or b1a3 would develop the knight\n- Some pawn pushes like f3f4 or c3c4 could be interesting\n\n\n- Central control and piece coordination seem crucial\n\nI'll need to carefully evaluate the positional benefits of each move, considering material balance, piece mobility, and potential threats. The decision isn't straightforward and requires nuanced strategic thinking.\n</think>\n\n<tool_call>\n<function=submit_answer>\n<parameter=answer>\ne1g1\n</parameter>\n</function>\n</tool_call>",
    },
]


tools = [
        {
            "type": "function",
            "function": {
                "name": "create_new_sub_agent",
                "description": "Create a fresh sub-agent (discarding any previous one) and give it this sub-task. The sub-agent starts with NO memory or context of this conversation, so the `text` must be fully self-contained: restate the complete position and everything needed to solve it.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "text": {
                            "type": "string",
                            "description": "A clear, fully self-contained description of the sub-task, including the complete position/board and the exact answer format required."
                        }
                    },
                    "required": [
                        "text"
                    ],
                    "additionalProperties": False
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "send_message_to_current_sub_agent",
                "description": "Continue with the current sub-agent (which remembers its earlier work) on a follow-up sub-task. Phrase it in terms of what it has already done.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "text": {
                            "type": "string",
                            "description": "A clear description of the follow-up sub-task for the current sub-agent."
                        }
                    },
                    "required": [
                        "text"
                    ],
                    "additionalProperties": False
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "submit_answer",
                "description": "Submit your final answer and finish the task. Call this exactly once, when you are done. The `answer` must be the final answer only, with no extra commentary, in the exact format the task requests.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "answer": {
                            "type": "string",
                            "description": "The final answer only, in the exact requested format (e.g. a single UCI move like e2e4)."
                        }
                    },
                    "required": [
                        "answer"
                    ],
                    "additionalProperties": False
                }
            }
        }
    ]


path = "../config_files/templates/Qwen3.5-4B.jinja"
manual_template = pathlib.Path(path).read_text(encoding="utf-8")

template = resolve_chat_template(MODEL_NAME, manual_template=manual_template)

# Now, we switch to the prefix-preserving chat-template and tokenize the conversation
tokenizer.chat_template = template

ids1 = tokenizer.apply_chat_template(
    [CONVERSATION],
    add_generation_prompt=True,
    tokenize=True,
    enable_thinking=True,
    return_tensors="pt",
    return_dict=True,
    tools=tools,
)

tids1 = ids1.input_ids[0]

ids2 = tokenizer.apply_chat_template(
    [CONVERSATION[:2]],
    add_generation_prompt=True,
    tokenize=True,
    enable_thinking=True,
    return_tensors="pt",
    return_dict=True,
    tools=tools,
)

tids2 = ids2.input_ids[0]

In [12]:
# verify that tids2 is prefix of tids1 and if not help me debug the error by printing the tokens and the differences
# clearly 
# i want you to print them side by side in a table format with the index, tids1 token, tids2 token, and the difference if any
# make sure the tokens are printed properly and only up to the shared suffix length of tids2
# and also that the table is properly formatted and easy to read
# i want the token values (decoded) as well for each token

def are_tokens_equal(tids1, tids2):
    if not tids1[:len(tids2)].equal(tids2):
        print("Error: tids2 is not a prefix of tids1. Here are the differences:")
        print(f"{'Index':<6} {'tids1 token':<20} {'tids1 decoded':<20} {'tids2 token':<20} {'tids2 decoded':<20} {'Difference':<10}")
        print("-" * 100)
        for i in range(len(tids2)):
            t1 = tids1[i].item()
            t2 = tids2[i].item()
            d1 = tokenizer.decode([t1])
            d2 = tokenizer.decode([t2])
            diff = "" if t1 == t2 else "<-- DIFFERENT"
            print(f"{i:<6} {t1:<20} {d1:<20} {t2:<20} {d2:<20} {diff:<10}")

In [15]:
# ok i have a theory
# i want you to add two consecutive user messages and then an assistant message
# my theory is that it removes the second consecutive user message
# please test this

conv = [
    {
        "role": "user",
        "content": "First user message.",
    },
    {
        "role": "user",
        "content": "Second user message.",
    },
]

# now lets test it as above

ids_test = tokenizer.apply_chat_template(
    [conv],
    add_generation_prompt=True,
    tokenize=True,
    enable_thinking=True,
    return_tensors="pt",
    return_dict=True,
    tools=tools,
)

ids_test2 = tokenizer.apply_chat_template(
    [conv[:1]],  # only the first three messages
    add_generation_prompt=True,
    tokenize=True,
    enable_thinking=True,
    return_tensors="pt",
    return_dict=True,
    tools=tools,
)


are_tokens_equal(ids_test.input_ids[0], ids_test2.input_ids[0])

Error: tids2 is not a prefix of tids1. Here are the differences:
Index  tids1 token          tids1 decoded        tids2 token          tids2 decoded        Difference
----------------------------------------------------------------------------------------------------
0      248045               <|im_start|>         248045               <|im_start|>                   
1      8678                 system               8678                 system                         
2      198                  
                    198                  
                              
3      2                    #                    2                    #                              
4      13455                 Tools               13455                 Tools                         
5      271                  

                   271                  

                             
6      2523                 You                  2523                 You                            
7      599        